# 03 — Conformance assessment

**Deliverables D8 (reactive sign), D9 (Volt-VAr) and D10 (Volt-Watt).**

Ports `build_conformance_voltvar.py` and `build_conformance_voltwatt.py` to DuckDB over
the local store. Every AS/NZS 4777.2 expression is imported from
`bms_sa_review.shared.as4777_curves` and runs unchanged — that is what makes these
results comparable to the Solar Analytics ones by construction rather than by inspection.

**D8 comes first, and not as a formality.** Under the sign convention as stored, the
three-phase cohort scores 81.7% reduced non-conformance dominated by `Q_adverse`.
Whether that is a real finding or a reporting artefact changes the headline result for
415 sites, so it is settled — or at least bounded — before anything is published.

Run `01_data_load.ipynb` and `02_fleet_eda.ipynb` first.

In [ ]:
# Reload edited .py modules without restarting the kernel.
# The library carries all the logic, so it gets edited constantly while notebooks
# stay thin -- without this, every module change means a kernel restart and a
# re-run of the expensive cells.
#
# CAVEAT: autoreload rebinds FUNCTIONS, not objects already constructed at import
# time. If you change a dataclass in se_params (SEAnalysisConfig, SEVoltVarParams)
# or a constant in se_config, restart the kernel -- `CONFIG` and `PARAMS` are
# module-level instances and will still be the old ones.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents)
     if (p / "solar_edge").is_dir() and (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from solar_edge.config import se_config as C
from solar_edge.lib import se_store, se_contract as contract, se_params
from solar_edge.lib import se_conformance as cf
from solar_edge.lib import se_sign as sign
from solar_edge.lib import se_plots as plots

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

con = se_store.connect()
config = se_params.CONFIG
params = se_params.PARAMS
display(contract.manifest(config, params).query("section != 'convention'"))

## D8 — The reactive-power sign

Settled for single-phase: SolarEdge reports in the **load convention** (positive =
absorbing), so ingest multiplies by −1. Locked in `se_config.REACTIVE_POWER_SIGN`.

Unsettled for three-phase. Two hypotheses, with opposite consequences:

- **(A)** three-phase inverters report reactive power with the **opposite polarity**, and
  flipping makes them behave like the single-phase cohort;
- **(B)** three-phase inverters genuinely respond in the **wrong direction**, which is a
  substantial conformance finding.

Three strands of evidence below. None settles it alone.

### D8.1 — Per-site response direction

A fleet median can be carried by a minority of strong responders, so the question is
better posed as *how many sites move which way*. Sites moving less than 0.02 kvar between
the low- and high-voltage bands are classed inactive rather than forced into a direction.

In [ ]:
classification = sign.site_response_classification(con, config)
display(pd.crosstab(
    classification.is_three_phase.map({False: "single-phase", True: "three-phase"}),
    classification.response_class,
))
print(f"Sites with enough data either side: {len(classification)} of 1,602")

### D8.2 — Where the deadband sits

The AS/NZS 4777.2 Australia A deadband runs 220–240 V. An inverter implementing the curve
shows minimum reactive **magnitude** inside it, rising on both sides.

This is the strand that distinguishes the two hypotheses. A cohort responding genuinely
backwards has no reason to reproduce the standard's own deadband geometry; a cohort
reporting inverted polarity reproduces it exactly, mirrored.

In [ ]:
shape = sign.deadband_shape(con, config)
for cohort, group in shape.groupby("cohort"):
    low = group.loc[group.median_abs_Q_kvar.idxmin()]
    print(f"{cohort:>13}: minimum |Q| = {low.median_abs_Q_kvar:.4f} kvar at {low.v_bin:.1f} V")
print(f"\nAS/NZS 4777.2 deadband: {C.as4777()['VVAR']['V2']:.0f}-{C.as4777()['VVAR']['V3']:.0f} V")
display(shape.pivot(index="v_bin", columns="cohort",
                    values=["median_Q_kvar", "median_abs_Q_kvar"]))

### D8.3 — What the decision costs

The full D9 scoring run on the three-phase cohort twice: as stored, and with `Q_kvar`
negated before any scoring, so the whole chain — tolerance band, capability clamp,
`Q_impact` — sees the flipped value.

If flipping moves the bulk of intervals out of `Q_adverse` and lands the cohort where the
single-phase one sits, that supports **(A)**. If it merely swaps one implausible picture
for another, **(B)** survives.

In [ ]:
flip = sign.sign_flip_sensitivity(con, config, params)
display(flip)
display(plots.plot_sign_flip(flip))

In [ ]:
display(sign.sign_evidence_summary(classification, shape, flip))

### D8 — where this lands

Read the three strands together, and note what each can and cannot support.

The **sign-flip sensitivity is the strongest strand**: flipping moves roughly 10.1 M
intervals out of `Q_adverse` and leaves the three-phase cohort within about two
percentage points of the single-phase cohort, with the same category profile — dominated
by significant shortfall rather than adverse response. Two independently-manufactured
populations landing on the same distribution after a sign change is not what hypothesis
(B) would predict.

The **deadband strand agrees**: both cohorts put their |Q| minimum at 230–235 V, inside
the standard's 220–240 V window.

The **per-site strand is weaker and should not be oversold** — only a minority of
three-phase sites have enough observations either side of the voltage split, and after
flipping they would still be more mixed than the single-phase cohort.

**This is evidence, not proof.** The decision needs SolarEdge documentation on
three-phase reactive-power sign reporting, or one site with known ground truth. Until it
arrives, D9 scores the cohorts **separately** and the three-phase result is reported with
this caveat attached rather than folded into a fleet number.

## D9 — Volt-VAr conformance

Interval scoring follows the CTE chain of `build_conformance_voltvar.build_sql`:
required Q → ±4% tolerance band → Figure 2.1 capability clamp → `Q_impact` → five
categories.

Two departures, both forced and both in the manifest:

- **Capacity basis is `s_99`**, not nameplate. It scales the required-Q curve, the
  tolerance band, the 20% assessability rule and the capability floor. Because `s_99` is
  an *observed* p99, a site that never approached its inverter limit gets a low `s_99`,
  a smaller required Q, and a flattering verdict. The bias has a direction; D15 sweeps it.
- **`capability_profile="review_corrected"`** — reactive-power priority above 0.8·S, and
  intervals below 0.2·S marked unassessable rather than scored, since Figure 2.1 sets no
  quantified minimum there.

`reduced_nonconf` = adverse + inactive + significant shortfall. `Q_near_conformant` is
excluded: those inverters deliver 90–110% of required reactive power. Milestone 3
included that band and excluded the shortfall band — an artefact of the swapped names
(R4) — so these figures will not match it, and should not.

In [ ]:
vvar_site_day = cf.voltvar_site_day(con, config, params)
print(f"{len(vvar_site_day):,} site-days scored")

vvar_summary = cf.voltvar_summary(vvar_site_day, by_cohort=True)
display(vvar_summary.T)
display(plots.plot_q_categories(vvar_summary))

### How strongly is the fleet responding?

The category counts say *how many* intervals fall outside the permitted band. This says
*by how much*. `Q_impact` is the measured response as a fraction of what the nearest
permitted band edge requires: 1.0 means sitting exactly on it, 0 means no response, below
0 means the wrong direction.

Read alongside the D6 finding that median power factor is 0.995–0.997 across both
cohorts. A fleet clustered near zero is responding weakly regardless of how the
categories tally.

In [ ]:
impact = cf.voltvar_impact_distribution(con, config, params)
display(plots.plot_q_impact_distribution(impact))

### Site-level verdicts

A site is conformant if its reduced non-conformance fraction is at or below 10%, matching
the Solar Analytics rule.

Sites with no capability-assessable intervals get **"not assessable"**, not "conformant".
Scoring an unobserved site as passing is how a conformance rate quietly becomes a measure
of coverage.

In [ ]:
vvar_verdicts = cf.voltvar_site_verdicts(vvar_site_day, config)
display(pd.crosstab(
    vvar_verdicts.is_three_phase.map({False: "single-phase", True: "three-phase"}),
    vvar_verdicts.verdict,
))
display(pd.crosstab(vvar_verdicts.state, vvar_verdicts.verdict))

## D10 — Volt-Watt conformance

A site is *exposed* above 253 V and non-conformant when measured P exceeds the Volt-Watt
ceiling plus the 4% tolerance.

**This is the "basic" variant only.** Without a counterfactual it detects generating
*above* the ceiling, but cannot distinguish an inverter that correctly curtailed from one
that simply had no sun. The GHI variant needs the D12 irradiance extract.

Note the population: D6 found only ~0.96% of intervals above 253 V. The exposed set is
thin, so report counts alongside percentages and treat site-level rates as noisy.

In [ ]:
vwatt_site_day = cf.voltwatt_site_day(con, config)
vwatt_summary = cf.voltwatt_summary(vwatt_site_day, by_cohort=True)
display(vwatt_summary.T)

vwatt_verdicts = cf.voltwatt_site_verdicts(vwatt_site_day, config)
display(pd.crosstab(
    vwatt_verdicts.is_three_phase.map({False: "single-phase", True: "three-phase"}),
    vwatt_verdicts.verdict,
))

### Population funnel

The gap between *exposed* and *assessable* is the point. A site can sit in the Volt-VAr
band all year and never be assessable, because below 20% of rated power Figure 2.1 sets
no quantified minimum capability. A conformance rate quoted without this table invites
the reader to assume the denominator is the whole fleet.

In [ ]:
display(cf.conformance_funnel(vvar_site_day, vwatt_site_day))

### A free corroboration

Volt-Watt exposure is the one place where the inverter-reported `derating_active` flag
and the standard's own ceiling should agree. If a large share of exposed intervals carry
the flag, the flag is at least partly grid-voltage-driven — which is the premise Method C
rests on at D14.

In [ ]:
exposed = vwatt_summary.exposed_intervals.sum()
flagged = vwatt_summary.exposed_with_derating_flag.sum()
print(f"Exposed intervals (V > 253 V):        {exposed:,}")
print(f"  of which derating_active is set:    {flagged:,}  ({100 * flagged / exposed:.1f}%)")
print()
print("Recall from D3 that the raw flag is 1.0 or NULL, never 0.0, so precision against")
print("it is interpretable but recall is not. D14 has to state that.")

## Persist the scored tables

Site-day grain, matching `conformance_voltvar_v2` / `conformance_voltwatt_v2` so the two
studies are directly comparable. Written to the store, not the repository.

In [ ]:
for name, frame in (("se_conformance_voltvar", vvar_site_day),
                    ("se_conformance_voltwatt", vwatt_site_day)):
    path = C.STORE_DIR / f"{name}.parquet"
    frame.to_parquet(path, index=False)
    print(f"{name}: {len(frame):,} rows -> {path}")

## What this establishes

- **Volt-VAr, single-phase:** 64.5% reduced non-conformance across 1,165 sites, dominated
  by *significant shortfall* — inverters responding, but far short of the required curve.
  Consistent with the D6 finding that fleet power factor sits at 0.995–0.997.
- **Volt-VAr, three-phase:** 81.7% as stored, 62.5% with the sign flipped. The sign
  decision is worth ~19 percentage points across 415 sites.
- **Volt-Watt:** thin exposure — 816 k intervals across 698 sites — with roughly 20%
  (single-phase) and 31% (three-phase) of exposed intervals above the ceiling. Wide
  intervals; treat as indicative.
- **62% of Volt-Watt-exposed intervals carry the derating flag** (507,568 of 816,066),
  which supports building Method C at D14.

## Open items

1. **Three-phase reactive sign** — needs SolarEdge documentation or ground truth. Until
   then cohorts are scored and reported separately.
2. **A latent bug in the original**, worth checking against your published figures:
   `build_conformance_voltwatt.py` (~line 160) builds `vw_exposed = "round(V,6) > 253.0"`
   then writes `CASE WHEN V > {vw_exposed}`, which expands to the chained comparison
   `V > round(V,6) > 253.0`. The GHI variant in the same file uses `WHEN {vw_exposed}`
   correctly. Not reproduced here.

## Next: D11 — Method A symptom scan

The apparent-limit scan over the 240–253 V band, seeded by the
`curtailment_eligible` counter already produced above.